# continuity

> Perform background matching to correct for discontinuities between detectors.

In [ ]:
# | default_exp euclid.continuity

In [ ]:
# | exporti

import logging

import numpy as np
from astropy.io import fits
from astropy.nddata import reshape_as_blocks
from astropy.wcs import WCS
from numpy.linalg import lstsq

from nicl.euclid.utilities import (
    get_dither_id_from_filename,
    get_obs_id_from_filename,
)
from nicl.mask import fast_mask

In [ ]:
# | hide
# additional imports for examples
%load_ext autoreload
%autoreload now -p -l

from pathlib import Path

from nicl.euclid.combine import combine
from nicl.euclid.constants import NISP
from nicl.euclid.utilities import default_data_path
from nicl.utilities import compute_pixel_scales

In [ ]:
# | export


def slice_to_coords(idx):
    """Given a tuple of two slice objects (as returned by numpy.index_exp), return an array of (x, y) index pairs.

    Parameters
    ----------
    idx : tuple of slice
        Tuple of two slice objects, typically (slice(y0, y1, step_y), slice(x0, x1, step_x)),
        representing the region of an image.

    Returns
    -------
    coords : ndarray of shape (N, 2)
        Array of (x, y) pixel coordinate pairs covering the region defined by the slices,
        where N is the total number of pixels in the region.

    Notes
    -----
    The output is a 2D array where each row is (x, y) for a pixel in the region.
    The order is row-major (y varies slowest).

    """
    ys, xs = idx  # unpack the two slices
    y0, y1, sy = ys.start or 0, ys.stop, ys.step or 1
    x0, x1, sx = xs.start or 0, xs.stop, xs.step or 1

    row_range = np.arange(y0, y1, sy)
    col_range = np.arange(x0, x1, sx)

    XX, YY = np.meshgrid(col_range, row_range)
    # stack into (rows, cols, 2) of (x, y)
    coords = np.stack((XX.ravel(), YY.ravel()), axis=-1)
    return coords


def can_wcs_shift_by_int(wcs1, wcs2):
    """Check if wcs1 and wcs2 are integer shiftable to match each other."""
    # 1) Must have the same number of axes
    if wcs1.wcs.naxis != wcs2.wcs.naxis:
        return False

    # 2) Match the reference world coord
    if not np.array_equal(wcs1.wcs.crval, wcs2.wcs.crval):
        return False

    # 3) Match the projection types & units
    if tuple(wcs1.wcs.ctype) != tuple(wcs2.wcs.ctype):
        return False
    if tuple(wcs1.wcs.cunit) != tuple(wcs2.wcs.cunit):
        return False

    # 4) Match the rotation+scale matrix (PC or CD)
    if hasattr(wcs1.wcs, "pc") and hasattr(wcs2.wcs, "pc"):
        if not np.array_equal(wcs1.wcs.pc, wcs2.wcs.pc):
            return False
        # If you’re using PC, you should also compare CDELT
        if not np.array_equal(wcs1.wcs.cdelt, wcs2.wcs.cdelt):
            return False
    else:
        # fallback to comparing CD (if present)
        if not np.array_equal(wcs1.wcs.cd, wcs2.wcs.cd):
            return False
    # 5) Check if the CRPIX offset is an integer
    crpix_offset = wcs1.wcs.crpix - wcs2.wcs.crpix
    for crpix in crpix_offset:
        if not crpix.is_integer():
            return False
    return True


def get_overlap_slices(hdr1, hdr2):
    """Return slice indices for the overlapping region of two images.

    Parameters
    ----------
    hdr1 : astropy.io.fits.Header
        The header of the first image
    hdr2 : astropy.io.fits.Header
        The header of the second image

    Returns
    -------
    tuple or None
        If the images can be aligned by an integer pixel shift and have an overlapping region,
        returns a tuple of two slice objects (numpy.index_exp):
        - The first slice extracts the overlapping region from the first image
        - The second slice extracts the corresponding region from the second image
        If the images do not overlap, returns (None, None).
        If the images cannot be integer-shifted to align with each other, raises a ValueError.

    """
    # extract CRPIX offsets
    w1, w2 = WCS(hdr1), WCS(hdr2)
    if not can_wcs_shift_by_int(w1, w2):
        raise ValueError(
            "The two images cannot be shifted by an integer to match each other."
        )
    cr1, cr2 = w1.wcs.crpix, w2.wcs.crpix
    dx, dy = int(cr2[0] - cr1[0]), int(cr2[1] - cr1[1])
    # image dimensions
    ny1, nx1 = hdr1["NAXIS2"], hdr1["NAXIS1"]
    ny2, nx2 = hdr2["NAXIS2"], hdr2["NAXIS1"]
    # in data1 frame data2 is shifted by (-dx, -dy)
    y0_1 = max(0, -dy)
    x0_1 = max(0, -dx)
    y1_1 = min(ny1, ny2 - dy)
    x1_1 = min(nx1, nx2 - dx)
    # check if any overlap at all
    if y0_1 >= y1_1 or x0_1 >= x1_1:
        return None, None
    # corresponding region in data2:
    y0_2 = y0_1 + dy
    x0_2 = x0_1 + dx
    y1_2 = y1_1 + dy
    x1_2 = x1_1 + dx
    return np.index_exp[y0_1:y1_1, x0_1:x1_1], np.index_exp[y0_2:y1_2, x0_2:x1_2]

In [ ]:
# | export


def bkg_match_corr(
    paths,  # list of paths to the images to be corrected
    corr_order=0,  # order of the correction (0, 1 or 2)
    binsize=None,  # bin size in pixels to form the equations
):
    logger = logging.getLogger(__name__)
    if corr_order not in (0, 1, 2):
        raise ValueError(f"corr_order must be 0, 1 or 2, got {corr_order} instead.")
    paths.sort()
    ids = []
    for path in paths:
        obs_id = get_obs_id_from_filename(path.name)
        dither_id = get_dither_id_from_filename(path.name)
        ids.append((obs_id, dither_id))
    a = []
    b = []
    solution_map = []
    match corr_order:
        case 0:
            n_unknowns_per_img = 1
        case 1:
            n_unknowns_per_img = 3
        case 2:
            n_unknowns_per_img = 5
    logger.info("Start building equations...")
    n_overlap_pairs = 0
    for i, path in enumerate(paths[:-1]):
        obs_id, dither_id = ids[i]
        # find out all images that belong to the other exposures and overlap with the current one
        other_paths = []
        for j, other_path in enumerate(paths[i + 1 :]):
            other_obs_id, other_dither_id = ids[i + j + 1]
            if other_obs_id != obs_id or other_dither_id != dither_id:
                logger.debug(
                    f"Checking overlap between {path.name} and {other_path.name}..."
                )
                hdr = fits.getheader(path, 0)
                other_hdr = fits.getheader(other_path, 0)
                slice1, slice2 = get_overlap_slices(hdr, other_hdr)
                # if they do overlap
                if slice1 is not None and slice2 is not None:
                    other_paths.append(other_path)
                    with (
                        fits.open(path, mode="update") as hdul,
                        fits.open(
                            path.with_suffix(".weight.fits"), mode="update"
                        ) as hdul_weight,
                        fits.open(other_path, mode="update") as other_hdul,
                        fits.open(
                            other_path.with_suffix(".weight.fits"),
                            mode="update",
                        ) as other_hdul_weight,
                    ):
                        # apply flux scaling
                        sci_hdr = hdul[0].header
                        flxscale = sci_hdr.get("FLXSCALE", 1.0)
                        if flxscale != 1.0:
                            hdul[0].data = hdul[0].data * flxscale
                            hdul[0].header["FLXSCALE"] = 1.0
                            hdul_weight[0].data = hdul_weight[0].data / flxscale**2
                            hdul_weight[0].header["FLXSCALE"] = 1.0
                        # apply flux scaling to the other image
                        other_sci_hdr = other_hdul[0].header
                        other_flxscale = other_sci_hdr.get("FLXSCALE", 1.0)
                        if other_flxscale != 1.0:
                            other_hdul[0].data = other_hdul[0].data * other_flxscale
                            other_hdul[0].header["FLXSCALE"] = 1.0
                            other_hdul_weight[0].data = (
                                other_hdul_weight[0].data / other_flxscale**2
                            )
                            other_hdul_weight[0].header["FLXSCALE"] = 1.0
                        with np.errstate(divide="ignore"):
                            diff_img_rms = np.sqrt(
                                1 / hdul_weight[0].data[slice1]
                                + 1 / other_hdul_weight[0].data[slice2]
                            )
                        idx_valid = np.isfinite(diff_img_rms)
                        npix_valid = np.sum(idx_valid)
                        if npix_valid == 0:
                            logger.debug(
                                "No valid pixels in the overlapping region, skipping."
                            )
                            continue
                        diff_img = other_hdul[0].data[slice2] - hdul[0].data[slice1]
                        # mask source pixels
                        if "mask" not in hdul:
                            logger.debug(
                                f"Object mask not found, building one for {path.name}..."
                            )
                            coverage_mask = (hdul[0].data == 0) & (
                                hdul_weight[0].data == 0
                            )
                            obj_mask = fast_mask(
                                hdul[0].data,
                                coverage_mask=coverage_mask,
                                estimate_background=True,
                                return_threshold=False,
                            )
                            # append the mask to the HDU list
                            hdul.append(
                                fits.ImageHDU(obj_mask.astype(np.uint8), name="mask")
                            )
                        else:
                            logger.debug(f"Object mask for {path.name} already built.")
                            obj_mask = hdul["mask"].data.astype(bool)
                        if "mask" not in other_hdul:
                            logger.debug(
                                f"Object mask not found, building one for {other_path.name}..."
                            )
                            other_coverage_mask = (other_hdul[0].data == 0) & (
                                other_hdul_weight[0].data == 0
                            )
                            other_obj_mask = fast_mask(
                                other_hdul[0].data,
                                coverage_mask=other_coverage_mask,
                                estimate_background=True,
                                return_threshold=False,
                            )
                            # append the mask to the HDU list
                            other_hdul.append(
                                fits.ImageHDU(
                                    other_obj_mask.astype(np.uint8), name="mask"
                                )
                            )
                        else:
                            logger.debug(
                                f"Object mask for {other_path.name} already built."
                            )
                            other_obj_mask = other_hdul["mask"].data.astype(bool)
                        # set source pixels to rms Inf (weight=0)
                        idx_source_combined = obj_mask[slice1] | other_obj_mask[slice2]
                        diff_img_rms[idx_source_combined] = np.inf
                        # check again if there are still any useful pixels
                        npix_valid = np.sum(np.isfinite(diff_img_rms))
                        if npix_valid == 0:
                            logger.debug(
                                "No useful pixels in the overlapping region after masking sources, skipping."
                            )
                            continue
                        # set up solution map
                        idx = paths.index(path)
                        idx_other = paths.index(other_path)
                        if idx in solution_map:
                            idx_solution = solution_map.index(idx)
                        else:
                            solution_map.append(idx)
                            idx_solution = len(solution_map) - 1
                        if idx_other in solution_map:
                            idx_other_solution = solution_map.index(idx_other)
                        else:
                            solution_map.append(idx_other)
                            idx_other_solution = len(solution_map) - 1
                        match corr_order:
                            case 0:
                                if binsize is None:
                                    binsize_ = diff_img.shape
                                else:
                                    binsize_ = (binsize, binsize)
                                # pad the image to be divisible by bin, don't wanna miss any pixels
                                diff_img = np.pad(
                                    diff_img,
                                    (
                                        (
                                            0,
                                            binsize_[0]
                                            - diff_img.shape[0] % binsize_[0],
                                        ),
                                        (
                                            0,
                                            binsize_[1]
                                            - diff_img.shape[1] % binsize_[1],
                                        ),
                                    ),
                                    mode="constant",
                                    constant_values=0,
                                )
                                diff_img_rms = np.pad(
                                    diff_img_rms,
                                    (
                                        (
                                            0,
                                            binsize_[0]
                                            - diff_img_rms.shape[0] % binsize_[0],
                                        ),
                                        (
                                            0,
                                            binsize_[1]
                                            - diff_img_rms.shape[1] % binsize_[1],
                                        ),
                                    ),
                                    mode="constant",
                                    constant_values=np.inf,
                                )
                                diff_img_blocks = reshape_as_blocks(diff_img, binsize_)
                                diff_img_rms_blocks = reshape_as_blocks(
                                    diff_img_rms, binsize_
                                )
                                with np.errstate(divide="ignore"):
                                    diff_img_rms_reduced_blocks = np.sqrt(
                                        1
                                        / np.sum(
                                            1 / diff_img_rms_blocks**2,
                                            axis=(2, 3),
                                        )
                                    )
                                idx_valid_blocks = np.isfinite(
                                    diff_img_rms_reduced_blocks
                                )
                                # remove blocks with no valid pixels
                                diff_img_rms_blocks = diff_img_rms_blocks[
                                    idx_valid_blocks
                                ]
                                diff_img_rms_reduced_blocks = (
                                    diff_img_rms_reduced_blocks[idx_valid_blocks]
                                )
                                diff_img_blocks = diff_img_blocks[idx_valid_blocks]
                                diff_img_reduced_blocks = np.average(
                                    diff_img_blocks,
                                    axis=(-2, -1),
                                    weights=1 / diff_img_rms_blocks**2,
                                )
                                coeff_array = np.zeros(
                                    (
                                        diff_img_reduced_blocks.size,
                                        n_unknowns_per_img * len(solution_map),
                                    )
                                )
                                coeff_array[:, idx_solution] = 1
                                coeff_array[:, idx_other_solution] = -1
                                offset = diff_img_reduced_blocks.ravel()
                                rms = diff_img_rms_reduced_blocks.ravel()
                                # weight the eqs by the inverse of the rms
                                coeff_array = coeff_array / rms[:, np.newaxis]
                                offset = offset / rms
                            case 1:
                                raise NotImplementedError(
                                    "First order correction is not implemented yet."
                                )
                            case 2:
                                raise NotImplementedError(
                                    "Second order correction is not implemented yet."
                                )
                        logger.debug(
                            f"Formed {len(offset)} equations from {path.name} vs. {other_path.name}"
                        )
                        a.append(coeff_array)
                        b.append(offset)
                else:
                    logger.debug(
                        f"{path.name} and {other_path.name} do not overlap, skipping."
                    )

        n_overlap_pairs += len(other_paths)
    n_imgs_in_eqs = len(solution_map)
    n_imgs = len(paths)
    logger.debug(
        f"Number of overlapping pairs: {n_overlap_pairs} out of {n_imgs} images."
    )
    imgs_not_in_eqs = []
    if n_imgs > n_imgs_in_eqs:
        idx_imgs_not_in_eqs = set(range(n_imgs)) - set(solution_map)
        logger.warning("The following images have no useful overlap with others:")
        for idx in idx_imgs_not_in_eqs:
            imgs_not_in_eqs.append(paths[idx])
            logger.warning(f"{paths[idx].name}")
    # padd zeros to coeff_array to match the longest one
    max_len = max([coeff.shape[1] for coeff in a])
    for i in range(len(a)):
        coeff = a[i]
        if coeff.shape[1] < max_len:
            coeff = np.pad(
                coeff, ((0, 0), (0, max_len - coeff.shape[1])), mode="constant"
            )
            a[i] = coeff
    # stack all coeff_array
    a = np.vstack(a, dtype=np.float32)
    # stack all offset
    b = np.hstack(b, dtype=np.float32)
    # solve the system of equations
    logger.info("Obtaining least-square solutions to the equations...")
    num_solutions = a.shape[1]
    logger.debug(f"number of equations: {len(a)}")
    logger.debug(f"number of unknowns: {num_solutions}")
    solutions, chi2, rank, _ = lstsq(a, b, rcond=None)
    if rank < num_solutions - 1:
        logger.warning(
            f"Rank deficiency more than 1: {rank=} < {num_solutions-1=}, which suggests that at least one group of images do not overlap with any other groups."
        )
    elif rank == num_solutions:
        logger.warning(
            f"{rank=} == {num_solutions=} by construction should never happen."
        )
    if rank < num_solutions or len(a) < num_solutions:
        chi2 = np.sum((b - a @ solutions) ** 2)
    chi2_reduced = chi2 / (len(b) - num_solutions)
    logger.debug(f"rank: {rank}, reduced chi^2: {chi2_reduced:.2f}")
    logger.debug(f"solutions map: {solution_map}")
    logger.debug(f"solutions: {solutions}")
    # remove the mask extension from the images
    logger.info("Removing the mask extension from the images...")
    for path in paths:
        with fits.open(path) as hdul:
            if "mask" in hdul:
                hdul.pop("mask")
                hdul.writeto(path, overwrite=True)
                logger.debug(f"Removed the mask extension from {path.name}.")
            else:
                logger.debug(f"No mask extension found in {path.name}.")
    # apply the solutions to the images
    logger.info("Applying the corrections to the images...")
    for idx_sol, idx in enumerate(solution_map):
        path = paths[idx]
        solution = solutions[
            idx_sol * n_unknowns_per_img : (idx_sol + 1) * n_unknowns_per_img
        ]
        logger.debug(f"Applying correction {solution} to {path.name}...")
        with (
            fits.open(path, mode="update") as hdul,
            fits.open(path.with_suffix(".weight.fits")) as hdul_weight,
        ):
            idx_not_blank = (hdul[0].data != 0) | (hdul_weight[0].data != 0)
            if corr_order == 0:
                hdul[0].data[idx_not_blank] += solution[0]
            elif corr_order == 1:
                # Apply a first-order polynomial correction to the image data
                # solution = [offset, coeff_x, coeff_y]
                ny, nx = hdul[0].data.shape
                y, x = np.indices((ny, nx))
                correction = solution[0] + solution[1] * x + solution[2] * y
                hdul[0].data[idx_not_blank] += correction[idx_not_blank]
            elif corr_order == 2:
                # Apply a second-order polynomial correction to the image data
                # solution = [offset, coeff_x, coeff_y, coeff_x2, coeff_y2]
                ny, nx = hdul[0].data.shape
                y, x = np.indices((ny, nx))
                correction = (
                    solution[0]
                    + solution[1] * x
                    + solution[2] * y
                    + solution[3] * x**2
                    + solution[4] * y**2
                )
                hdul[0].data[idx_not_blank] += correction[idx_not_blank]
            hdul.flush()
    return imgs_not_in_eqs

## Examples

### generate mock data for testing

Four mock NISP dithers with real WCS from obsid=2683 and fake background at ~100 ADU. Perhaps inject fake sources as well.

In [ ]:
dithers_dir = default_data_path("Q1_R1", "NIR", "2683")
dither_paths = dithers_dir.glob("EUC_NIR_W-CAL-IMAGE_H-2683-*.fits")
mock_dithers_dir = default_data_path("Q1_R1_mock", "continuity", "sky")
sky_ped = 100.0
sky_rms = 5.0
chip_discon = 1.0
for path in dither_paths:
    dither_id = get_dither_id_from_filename(path)
    with fits.open(path) as hdul:
        hdul_ = [hdul[0]]
        for ext in NISP.extnames:
            hdr = hdul[ext + ".SCI"].header
            mag_zpt = hdr.get("ZPAB")
            nx, ny = hdr["NAXIS1"], hdr["NAXIS2"]
            pix_scl_eff = compute_pixel_scales(hdr)
            # shit the sky level by 1 adu every dither
            img = np.random.normal(sky_ped + int(dither_id), sky_rms, (ny, nx))
            # introduce some discontinuity as sb
            img += np.random.choice([-chip_discon, chip_discon])
            # acount for variations in pixel scale and magnitude zero point
            img *= (pix_scl_eff / 0.3) ** 2 * 10 ** (0.4 * (mag_zpt - 30))
            # introduce some discontinuity as raw counts
            # img += np.random.choice([-chip_discon, chip_discon])
            # create rms extension
            rms = np.full((ny, nx), sky_rms)
            rms *= (pix_scl_eff / 0.3) ** 2 * 10 ** (0.4 * (mag_zpt - 30))
            rms_hdr = hdul[ext + ".RMS"].header
            # create dq extension
            dq = np.zeros((ny, nx), dtype=np.uint8)
            dq_hdr = hdul[ext + ".DQ"].header
            hdul_.append(fits.ImageHDU(img, header=hdr))
            hdul_.append(fits.ImageHDU(rms, header=rms_hdr))
            hdul_.append(fits.ImageHDU(dq, header=dq_hdr))
        hdul_ = fits.HDUList(hdul_)
        out_path = (
            mock_dithers_dir / f"EUC_NIR_W-CAL-IMAGE_H-2683-{dither_id}_mock.fits"
        )
        out_path.parent.mkdir(parents=True, exist_ok=True)
        hdul_.writeto(out_path, overwrite=True)

In [ ]:
from nicl import configure_logging

configure_logging(level="DEBUG")
configure_logging(name="nicl.mask", level="WARNING")

In [ ]:
in_dir = mock_dithers_dir
out_dir = default_data_path("Q1_R1_processed_test", "continuity")
combine(
    in_dir=in_dir,
    out_dir=out_dir,
    obs_ids=2683,
    filters="H",
    bkg_sub=False,
    bkg_match=True,
    name="2683_bkg_match",
    overwrite=True,
)

In [ ]:
in_dir = default_data_path(
    "Q1_R1_processed_v0.7",
    "persistence",
    "NIR",
)
out_dir = Path("~", "Q1_R1_processed_test")
ids = [2683, 2717, 3588, 3628]
for id in ids:
    combine(
        in_dir=in_dir,
        out_dir=out_dir,
        obs_ids=id,
        filters="H",
        bkg_sub=False,
        bkg_match=True,
        name=f"{id}_bkg_match",
        pixel_scale=0.3,
        overwrite=True,
    )

In [ ]:
in_dir = default_data_path(
    "Q1_R1",
    "VIS_QUAD",
)
out_dir = Path("~", "Q1_R1_processed_test")
combine(
    in_dir=in_dir,
    out_dir=out_dir,
    obs_ids=2683,
    filters="I",
    bkg_sub=False,
    autodark_corr=True,
    autodark_dir=default_data_path("Q1_R1_processed_v0.7", "skyflat", "VIS"),
    bkg_match=True,
    name="2683_bkg_match",
    pixel_scale=0.3,
    overwrite=True,
)

In [ ]:
# | hide
import nbdev

nbdev.nbdev_export()